In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [11]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [10]:
df = df.fillna(0)

### Preparing the dataset 

Preparation:

* Fill missing values with zeros.
* Do the split with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
```

* Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [15]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [18]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

In [22]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [23]:
df_train

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration
0,2011,USA,Gasoline,Front-wheel drive,4,2370,6,284.0,4280,19.0
1,2012,USA,Diesel,All-wheel drive,3,2460,6,288.0,4580,18.8
2,1985,Europe,Gasoline,All-wheel drive,3,2070,6,230.0,4150,20.6
3,2005,Europe,Diesel,Front-wheel drive,5,2050,5,249.0,4120,0.0
4,2004,Asia,Gasoline,Front-wheel drive,2,2160,6,241.0,4020,17.9
...,...,...,...,...,...,...,...,...,...,...
5995,1988,Asia,Gasoline,Rear-wheel drive,2,2290,6,235.0,4110,17.9
5996,2009,Asia,Gasoline,Rear-wheel drive,3,2400,6,245.0,4460,17.5
5997,2012,Asia,Gasoline,Front-wheel drive,3,2510,6,271.0,4440,19.1
5998,2003,Europe,Gasoline,All-wheel drive,4,1900,5,211.0,4000,19.4


## Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable. 

* Train a model with `max_depth=1`.


Which original feature is used for splitting the data? If you use
`DictVectorizer`, map a one-hot feature such as `origin=USA` back to its
original column name.


* `'vehicle_weight'`
* `'model_year'`
* `'origin'`
* `'fuel_type'`

In [26]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(
    df_train.to_dict(orient='records')
)

In [27]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1)

In [28]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Question 2

Train a random forest regressor with these parameters:

* `n_estimators=10`
* `random_state=1`
* `n_jobs=-1` (optional - to make training faster)


What's the RMSE of this model on the validation data?

* 0.1837
* 1.837
* 18.37
* 183.7

In [29]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error


X_val = dv.transform(df_val.to_dict(orient='records'))

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
print(root_mean_squared_error(y_val, y_pred))

1.837054054730018


## Question 3

Now let's experiment with the `n_estimators` parameter

* Try these values of this parameter: `[10, 50, 100, 150]`.
* Set `random_state` to `1`.
* Evaluate the model on the validation dataset.


Which value gives the lowest validation RMSE? Consider 3 decimal places for
calculating the answer.

- 10
- 50
- 100
- 150

In [30]:
n_estimators_list = [10, 50, 100, 150]

for n_estimator in n_estimators_list:
    rf = RandomForestRegressor(n_estimators=n_estimator, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_val)
    print(root_mean_squared_error(y_val, y_pred))

1.837054054730018
1.7715666428333987
1.7681362324492986
1.7688040324216567


## Question 4

Let's select the best `max_depth`:

* Try different values of `max_depth`: `[10, 15, 20, 25]`
* For each of these values,
  * try different values of `n_estimators` from `[10, 50, 100, 150]`
  * calculate the mean RMSE 
* Fix the random seed: `random_state=1`


What's the best `max_depth`, using the mean RMSE?

* 10
* 15
* 20
* 25

In [33]:
import numpy as np

for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(root_mean_squared_error(y_val, rf.predict(X_val)))
    print(d, np.mean(scores))

10 1.7554047863550184
15 1.7839526938716295
20 1.7882420685694518
25 1.786317363879773


# Question 5

We can extract feature importance information from tree-based models. 

At each step of the decision tree learning algorithm, it finds the best split. 
When doing it, we can calculate "gain" - the reduction in impurity before and after the split. 
This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the
[`feature_importances_`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.feature_importances_)
field. 

For this homework question, we'll find the most important feature:

* Train the model with these parameters:
  * `n_estimators=10`,
  * `max_depth=20`,
  * `random_state=1`,
  * `n_jobs=-1` (optional)
* Get the feature importance information from this model


What's the most important feature (among these 4)? 

* `vehicle_weight`
*	`horsepower`
* `acceleration`
* `engine_displacement`	

In [37]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out())
print(importances.sort_values(ascending=False))

model_year                      0.296001
vehicle_weight                  0.203879
fuel_type=Gasoline              0.160712
horsepower                      0.078017
engine_displacement             0.061005
acceleration                    0.055201
origin=USA                      0.042144
drivetrain=Front-wheel drive    0.020145
num_doors                       0.019415
fuel_type=Diesel                0.017038
fuel_type=Hybrid                0.015800
drivetrain=All-wheel drive      0.007682
num_cylinders                   0.007284
origin=Asia                     0.006391
drivetrain=Rear-wheel drive     0.005781
origin=Europe                   0.003505
dtype: float64


## Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

* Install XGBoost
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:

```
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?

* 0.3
* 0.1
* Both give equal value

In [50]:
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=10)
    print(eta, root_mean_squared_error(y_val, model.predict(dval)))

[0]	train-rmse:2.40653	val-rmse:2.46782
[10]	train-rmse:1.47125	val-rmse:1.73205
[20]	train-rmse:1.39368	val-rmse:1.73766
[30]	train-rmse:1.33243	val-rmse:1.74548
[40]	train-rmse:1.25326	val-rmse:1.76296
[50]	train-rmse:1.18645	val-rmse:1.77517
[60]	train-rmse:1.10265	val-rmse:1.79401
[70]	train-rmse:1.04604	val-rmse:1.80444
[80]	train-rmse:0.99487	val-rmse:1.81308
[90]	train-rmse:0.94986	val-rmse:1.82135
[99]	train-rmse:0.90106	val-rmse:1.82646
0.3 1.8264579799904583
[0]	train-rmse:2.74739	val-rmse:2.78635
[10]	train-rmse:1.80592	val-rmse:1.93152
[20]	train-rmse:1.55847	val-rmse:1.75076
[30]	train-rmse:1.47544	val-rmse:1.71667
[40]	train-rmse:1.43074	val-rmse:1.71289
[50]	train-rmse:1.40368	val-rmse:1.71358
[60]	train-rmse:1.37884	val-rmse:1.71688
[70]	train-rmse:1.36283	val-rmse:1.71919
[80]	train-rmse:1.34578	val-rmse:1.72100
[90]	train-rmse:1.32546	val-rmse:1.72303
[99]	train-rmse:1.31192	val-rmse:1.72481
0.1 1.724810059827976
